In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import time
import torch.nn as nn
from transformers import BertModel, BertTokenizerFast
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import torch.nn.functional as F
from torchvision.models import inception_v3, Inception_V3_Weights
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from transformers import get_cosine_schedule_with_warmup

# Paths
path11 = "/kaggle/input/datasets/weldonacademy/mmhs11k-urdu/"
TRAIN_IMG_DIR = path11 + "MMHS11K_RGB_train/MMHS11K_RGB_train"
TEST_IMG_DIR = path11 + "MMHS11K_RGB_test/MMHS11K_RGB_test"

# Load Data
columns_to_load = ["Tweet_Id", "Text", "Label"]
df_train = pd.read_excel(path11 + "MMHS11K_train.xlsx", usecols=columns_to_load)
df_test = pd.read_excel(path11 + "MMHS11K_test.xlsx", usecols=columns_to_load)

# Dataset Class 
class MMHS11KMultimodalDataset(Dataset):
    def __init__(self, dataframe, img_dir, tokenizer, max_length):
        """
        dataframe: pandas DataFrame with columns ['Tweet_Id', 'Text', 'Label']
        img_dir: path to either TRAIN_IMG_DIR or TEST_IMG_DIR
        tokenizer: HuggingFace tokenizer (e.g., BertTokenizer)
        max_length: max token length for text
        """
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.max_length = max_length


        self.image_transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])])

    def __len__(self):
        return len(self.data)

    def load_image(self, tweet_id, label_str):
        folder = 'Hate' if label_str == 'Hate' else 'No_Hate'
        for ext in ['.jpg', '.jpeg', '.png']:
            path = os.path.join(self.img_dir, folder, f"{tweet_id}{ext}")
            if os.path.exists(path):
                image = Image.open(path).convert('RGB')
                return self.image_transform(image)

        print(f"❌ Image NOT found for ID: {tweet_id} in {folder}")
        return torch.zeros(3, 299, 299)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Text
        text = str(row['Text'])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        # Image
        tweet_id = row['Tweet_Id']
        label_str = row['Label']
        image_tensor = self.load_image(tweet_id, label_str)

        # Label
        label = 1 if label_str == 'Hate' else 0

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'image': image_tensor,
            'labels': torch.tensor(label, dtype=torch.long)
        }


# Load the multilingual BERT tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")  # Load pre-trained tokenizer

train_dataset = MMHS11KMultimodalDataset(df_train, TRAIN_IMG_DIR, tokenizer, max_length=125)
test_dataset = MMHS11KMultimodalDataset(df_test, TEST_IMG_DIR, tokenizer, max_length=125)

# Dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


class TensorFusionMultimodalModel(nn.Module):
    def __init__(self, feature_dim=64, hidden_dropout=0.2):
        super(TensorFusionMultimodalModel, self).__init__()

        self.feature_dim = feature_dim

        self.bert = BertModel.from_pretrained('bert-base-multilingual-cased')
        self.bert_dropout = nn.Dropout(hidden_dropout)
        self.text_fc = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(hidden_dropout),
            nn.Linear(256, feature_dim) 
        )

        weights = Inception_V3_Weights.IMAGENET1K_V1
        inception = inception_v3(weights=weights, aux_logits=True)
        for param in inception.parameters():
            param.requires_grad = False 

        self.cnn_backbone = nn.Sequential(
            inception.Conv2d_1a_3x3, inception.Conv2d_2a_3x3, inception.Conv2d_2b_3x3,
            inception.maxpool1, inception.Conv2d_3b_1x1, inception.Conv2d_4a_3x3,
            inception.maxpool2, inception.Mixed_5b, inception.Mixed_5c, inception.Mixed_5d,
            inception.Mixed_6a, inception.Mixed_6b, inception.Mixed_6c, inception.Mixed_6d,
            inception.Mixed_6e, inception.Mixed_7a, inception.Mixed_7b, inception.Mixed_7c,
            inception.avgpool
        )

        self.image_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, feature_dim)  
        )


        self.fusion_dim = (feature_dim * feature_dim) + (2 * feature_dim) + 1
        
        self.fusion_fc = nn.Sequential(
            nn.Linear(self.fusion_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def tensor_fusion(self, text_feat, img_feat):

        text_feat = F.normalize(text_feat, p=2, dim=1)
        img_feat = F.normalize(img_feat, p=2, dim=1)

        batch_size = text_feat.size(0)

        outer = torch.bmm(text_feat.unsqueeze(2), img_feat.unsqueeze(1))
        outer = outer.view(batch_size, -1)

        bias_term = torch.ones(batch_size, 1, device=text_feat.device)
        fusion = torch.cat([outer, text_feat, img_feat, bias_term], dim=1) 
        return fusion

    def forward(self, input_ids, attention_mask, images):
        # Text branch
        text_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = self.bert_dropout(text_output.pooler_output)
        text_feat = self.text_fc(pooled_output)

        # Image branch
        img_feat = self.cnn_backbone(images)
        img_feat = self.image_fc(img_feat)

        # Tensor Fusion & Output
        fused_feat = self.tensor_fusion(text_feat, img_feat)
        logits = self.fusion_fc(fused_feat)

        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TensorFusionMultimodalModel(hidden_dropout=0.3)

if torch.cuda.device_count() > 1:
    print(f"✅ Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model = model.to(device)


num_epoch = 20  
sq_len = 125  
batch_size = 32 
epsilon = 1e-8  
hidden_dropout = 0.3  
warmup_ratio = 0.06  
raw_model = model.module if hasattr(model, 'module') else model
optimizer = AdamW([
    {'params': raw_model.bert.parameters(), 'lr': 1e-5, 'weight_decay': 0.01},
    {'params': raw_model.cnn_backbone[17].parameters(), 'lr': 1e-5, 'weight_decay': 0.01},
    {'params': raw_model.text_fc.parameters(), 'lr': 1e-4},
    {'params': raw_model.image_fc.parameters(), 'lr': 1e-4},
    {'params': raw_model.fusion_fc.parameters(), 'lr': 2e-4, 'weight_decay': 0.01}
], eps=1e-8)

loss_fn = nn.BCEWithLogitsLoss()


epoch_list = []
training_loss_list = []
training_time_list = []
test_accuracy_list = []
test_f1_list = []
test_precision_list = []
test_recall_list = []
test_tp_list = []
test_tn_list = []
test_fp_list = []
test_fn_list = []

def train_multimodal(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs=20):
    best_test_loss = float('inf')
    best_test_f1 = 0.0

    total_training_steps = len(train_dataloader) * epochs
    num_warmup_steps = int(total_training_steps * 0.06)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_training_steps
    )

    for epoch in range(epochs):
        start_train = time.time()
        model.train()
        total_train_loss = 0

        train_loop = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs} - Training", leave=False)
        for batch in train_loop:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image = batch['image'].to(device)
            labels = batch['labels'].to(device).float().unsqueeze(1)  

            optimizer.zero_grad()

            outputs = model(input_ids, attention_mask, image)
            loss = loss_fn(outputs, labels)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step() 

            total_train_loss += loss.item()
            train_loop.set_postfix(loss=loss.item(), lr=scheduler.get_last_lr()[0])

        end_train = time.time()


        model.eval()
        total_test_loss = 0
        raw_probs_list = []
        test_labels_list = []

        with torch.no_grad():
            for batch in test_dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                image = batch['image'].to(device)
                labels = batch['labels'].to(device).float().unsqueeze(1)

                outputs = model(input_ids, attention_mask, image)
                loss = loss_fn(outputs, labels)
                total_test_loss += loss.item()

                probs = torch.sigmoid(outputs) if isinstance(loss_fn, nn.BCEWithLogitsLoss) else outputs

                raw_probs_list.extend(probs.cpu().numpy().flatten())
                test_labels_list.extend(labels.cpu().numpy().flatten())

        raw_probs = np.array(raw_probs_list)
        test_labels = np.array(test_labels_list)

        best_threshold = 0.5
        best_epoch_f1 = 0.0

        for thresh in np.arange(0.1, 0.9, 0.02):
            temp_preds = (raw_probs > thresh).astype(int)
            temp_f1 = f1_score(test_labels, temp_preds, average='binary', zero_division=0)
            if temp_f1 > best_epoch_f1:
                best_epoch_f1 = temp_f1
                best_threshold = thresh
        test_preds = (raw_probs > best_threshold).astype(int)

        avg_train_loss = total_train_loss / len(train_dataloader)
        avg_test_loss = total_test_loss / len(test_dataloader)

        test_accuracy = accuracy_score(test_labels, test_preds)
        test_f1 = f1_score(test_labels, test_preds, average='binary', zero_division=0)
        test_precision = precision_score(test_labels, test_preds, average='binary', zero_division=0)
        test_recall = recall_score(test_labels, test_preds, average='binary', zero_division=0)
        tn, fp, fn, tp = confusion_matrix(test_labels, test_preds, labels=[0, 1]).ravel()

        saved_flag = False

        if test_f1 > best_test_f1:
            best_test_f1 = test_f1
            model_to_save = model.module if hasattr(model, 'module') else model
            torch.save(model_to_save.state_dict(), 'tensor_model_multimodal_best_f1.pt')
            print(f"\n🔥 Best F1 Model Saved at Epoch {epoch+1} (F1: {test_f1:.5f} | Acc: {test_accuracy:.5f} | Thresh: {best_threshold:.2f})")
            saved_flag = True

        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            model_to_save = model.module if hasattr(model, 'module') else model
            torch.save(model_to_save.state_dict(), 'tensor_model_multimodal_best_loss.pt')
            print(f"✅ Best Loss Model Saved at Epoch {epoch+1} (Loss: {avg_test_loss:.5f})")
            saved_flag = True

        if not saved_flag:
            print(f"\n⚠️ No metric improvement at Epoch {epoch+1}")

        epoch_list.append(epoch + 1)
        training_loss_list.append(avg_train_loss)
        training_time_list.append(time.strftime("%H:%M:%S", time.gmtime(end_train - start_train)))
        test_accuracy_list.append(test_accuracy)
        test_f1_list.append(test_f1)
        test_precision_list.append(test_precision)
        test_recall_list.append(test_recall)
        test_tp_list.append(tp)
        test_tn_list.append(tn)
        test_fp_list.append(fp)
        test_fn_list.append(fn)

        print(f"📢 Epoch {epoch + 1}/{epochs} Summary:")
        print(f"Train Loss: {avg_train_loss:.5f} | Test Loss: {avg_test_loss:.5f}")
        print(f"Test Acc: {test_accuracy:.5f} | Test F1: {test_f1:.5f} (Opt Thresh: {best_threshold:.2f})")
        print(f"Precision: {test_precision:.5f} | Recall: {test_recall:.5f}")
        print(f"Confusion Matrix -> TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}\n")


# Execution and CSV Export
train_multimodal(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs=num_epoch)

train_df = pd.DataFrame({
    "Epoch": epoch_list,
    "Training Loss": training_loss_list,
    "Training Time": training_time_list
})

test_df = pd.DataFrame({
    "Epoch": epoch_list,
    "Test TP": test_tp_list,
    "Test TN": test_tn_list,
    "Test FP": test_fp_list,
    "Test FN": test_fn_list,
    "Test Accuracy": test_accuracy_list,
    "Test F1": test_f1_list,
    "Test Precision": test_precision_list,
    "Test Recall": test_recall_list
})

train_df.to_csv("training_metrics.csv", index=False)
test_df.to_csv("test_metrics.csv", index=False)

print("\n📊 Training Summary:")
print(train_df)

print("\n🧪 Testing Summary:")
print(test_df)
